In [17]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [18]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
%cd /content
!mkdir -p /content/data
%cd /content/data

# Download Task03_Liver
!wget -O Task03_Liver.tar https://msd-for-monai.s3-us-west-2.amazonaws.com/Task03_Liver.tar
!tar -xf Task03_Liver.tar
!ls -lah /content/data/Task03_Liver

In [ ]:
!pip -q install nibabel monai opencv-python matplotlib tqdm

In [ ]:
!pip -q install git+https://github.com/facebookresearch/segment-anything.git

In [ ]:
import torch, sys
print(torch.__version__)
print("CUDA:", torch.cuda.is_available())

In [ ]:
import os
import numpy as np
import nibabel as nib

images_dir = "/content/data/Task03_Liver/imagesTr"
labels_dir = "/content/data/Task03_Liver/labelsTr"

img_files = sorted([os.path.join(images_dir, f) for f in os.listdir(images_dir) if f.endswith(".nii") or f.endswith(".nii.gz")])
lab_files = sorted([os.path.join(labels_dir, f) for f in os.listdir(labels_dir) if f.endswith(".nii") or f.endswith(".nii.gz")])

print("Images:", len(img_files), "Labels:", len(lab_files))
print("Example:", img_files[0], lab_files[0])

In [ ]:
def window_ct_hu(slice_hu, level=50, width=400):
    low = level - width/2
    high = level + width/2
    x = np.clip(slice_hu, low, high)
    x = (x - low) / (high - low) * 255.0
    return x.astype(np.uint8)

def to_rgb(gray_u8):
    return np.stack([gray_u8, gray_u8, gray_u8], axis=-1)  # H,W,3

In [ ]:
def mask_to_box(mask2d):
    ys, xs = np.where(mask2d > 0)
    if len(xs) == 0:
        return None
    x1, x2 = xs.min(), xs.max()
    y1, y2 = ys.min(), ys.max()
    return np.array([x1, y1, x2, y2])

In [ ]:
!mkdir -p /content/checkpoints
!wget -O /content/checkpoints/sam_vit_b_01ec64.pth https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

In [ ]:
!mkdir -p /content/checkpoints
!wget -O /content/checkpoints/sam_vit_b_01ec64.pth https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

In [ ]:
import torch
from segment_anything import sam_model_registry, SamPredictor

device = "cuda" if torch.cuda.is_available() else "cpu"
sam = sam_model_registry["vit_b"](checkpoint="/content/checkpoints/sam_vit_b_01ec64.pth")
sam.to(device=device)
predictor = SamPredictor(sam)

def run_sam_on_slice(rgb_img, box_xyxy):
    predictor.set_image(rgb_img)
    masks, scores, logits = predictor.predict(
        box=box_xyxy[None, :],  # shape (1,4)
        multimask_output=False
    )
    return masks[0].astype(np.uint8)

In [ ]:
def dice(pred, gt, eps=1e-8):
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    inter = (pred & gt).sum()
    return (2*inter + eps) / (pred.sum() + gt.sum() + eps)

def iou(pred, gt, eps=1e-8):
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    inter = (pred & gt).sum()
    union = (pred | gt).sum()
    return (inter + eps) / (union + eps)

In [ ]:
from tqdm import tqdm

all_dice = []
all_iou = []
fail_thresh = 0.5

for img_path, lab_path in tqdm(list(zip(img_files, lab_files))):
    img = nib.load(img_path).get_fdata()  # (H,W,Z) or (H,W,D)
    lab = nib.load(lab_path).get_fdata()

    # Ensure same shape
    assert img.shape == lab.shape, (img.shape, lab.shape)

    # Loop axial slices along last axis
    for z in range(img.shape[-1]):
        gt = (lab[..., z] > 0).astype(np.uint8)
        if gt.sum() == 0:
            continue  # non-empty slices only

        box = mask_to_box(gt)
        if box is None:
            continue

        gray = window_ct_hu(img[..., z], level=50, width=400)
        rgb = to_rgb(gray)

        pred = run_sam_on_slice(rgb, box)

        d = dice(pred, gt)
        j = iou(pred, gt)

        all_dice.append(d)
        all_iou.append(j)

print("Slices evaluated:", len(all_dice))
print("Mean Dice:", float(np.mean(all_dice)))
print("Mean IoU:", float(np.mean(all_iou)))
print("Failure rate:", float(np.mean(np.array(all_dice) < fail_thresh)))

In [ ]:
images_dir = "/content/data/Task03_Liver/imagesTr"
labels_dir = "/content/data/Task03_Liver/labelsTr"

img_files = sorted([
    os.path.join(images_dir, f)
    for f in os.listdir(images_dir)
    if f.endswith(".nii.gz") and not f.startswith("._")
])

lab_files = sorted([
    os.path.join(labels_dir, f)
    for f in os.listdir(labels_dir)
    if f.endswith(".nii.gz") and not f.startswith("._")
])

print("Images:", len(img_files))
print("Labels:", len(lab_files))
print("Example:", img_files[0])

In [ ]:
!find /content/data/Task03_Liver -name "._*" -delete

In [ ]:
!ls /content/data/Task03_Liver/imagesTr | head

In [ ]:
for img_path, lab_path in tqdm(list(zip(img_files, lab_files))):
    img = nib.load(img_path).get_fdata()
    lab = nib.load(lab_path).get_fdata()

In [ ]:
img = nib.load(img_files[0]).get_fdata()
lab = nib.load(lab_files[0]).get_fdata()

print(img.shape, lab.shape)

In [ ]:
import numpy as np
import nibabel as nib
from tqdm import tqdm
import os

def collect_nonempty_slices(img_files, lab_files, max_slices=None, seed=0):
    """
    Returns a list of (img_path, lab_path, z_index) only for slices where GT>0.
    If max_slices is set, subsamples uniformly for speed/fairness.
    """
    triples = []
    for img_path, lab_path in tqdm(list(zip(img_files, lab_files)), desc="Indexing non-empty slices"):
        img = nib.load(img_path)
        lab = nib.load(lab_path).get_fdata()
        for z in range(lab.shape[-1]):
            if (lab[..., z] > 0).sum() > 0:
                triples.append((img_path, lab_path, z))

    if max_slices is not None and len(triples) > max_slices:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(triples), size=max_slices, replace=False)
        triples = [triples[i] for i in idx]

    return triples

In [ ]:
PERTURBATIONS = {
    "Noise(sigma)": [0, 5, 10, 20, 30],
    "Blur(k)":      [0, 3, 5, 7, 9],
    "DownUp(scale)":[1.0, 0.75, 0.5, 0.35, 0.25],
    "Gamma":        [1.0, 0.8, 0.6, 1.2, 1.5],
    "Contrast(a)":  [1.0, 0.8, 0.6, 1.2, 1.5],
}

In [ ]:
def apply_perturb(gray_u8, ptype, val):
    if ptype == "Noise(sigma)":
        return gray_u8 if val == 0 else add_gaussian_noise(gray_u8, sigma=val)
    if ptype == "Blur(k)":
        return gray_u8 if val == 0 else gaussian_blur(gray_u8, k=int(val))
    if ptype == "DownUp(scale)":
        return gray_u8 if val == 1.0 else down_up(gray_u8, scale=float(val))
    if ptype == "Gamma":
        return gray_u8 if val == 1.0 else gamma_correction(gray_u8, gamma=float(val))
    if ptype == "Contrast(a)":
        return gray_u8 if val == 1.0 else contrast_scale(gray_u8, alpha=float(val))
    raise ValueError("Unknown perturbation type")

In [ ]:
def eval_organ_curves(triples, organ_name):
    curves = {}  # ptype -> list of (severity, mean_dice)
    for ptype, severities in PERTURBATIONS.items():
        mean_dices = []

        for sev in severities:
            dices = []
            for (img_path, lab_path, z) in tqdm(triples, desc=f"{organ_name} | {ptype}={sev}", leave=False):
                img = nib.load(img_path).get_fdata()
                lab = nib.load(lab_path).get_fdata()

                gt = (lab[..., z] > 0).astype(np.uint8)
                box = mask_to_box(gt)
                if box is None:
                    continue

                gray = window_ct_hu(img[..., z], level=50, width=400)
                gray_p = apply_perturb(gray, ptype, sev)
                rgb = to_rgb(gray_p)

                pred = run_sam_on_slice(rgb, box)
                dices.append(dice(pred, gt))

            mean_dices.append(float(np.mean(dices)) if len(dices) else np.nan)

        curves[ptype] = (severities, mean_dices)

    return curves

In [ ]:
import matplotlib.pyplot as plt

def plot_spleen_vs_liver(spleen_curves, liver_curves, outpath="spleen_vs_liver_robustness.png"):
    ptypes = list(PERTURBATIONS.keys())
    n = len(ptypes)

    fig, axes = plt.subplots(1, n, figsize=(4*n, 3.2), sharey=True)
    if n == 1:
        axes = [axes]

    for ax, ptype in zip(axes, ptypes):
        xs_s, ys_s = spleen_curves[ptype]
        xs_l, ys_l = liver_curves[ptype]

        ax.plot(xs_s, ys_s, marker="o", label="Spleen")
        ax.plot(xs_l, ys_l, marker="o", label="Liver")
        ax.set_title(ptype)
        ax.set_xlabel("Severity")
        ax.grid(True, alpha=0.3)

    axes[0].set_ylabel("Mean Dice")
    axes[-1].legend(loc="best")

    plt.tight_layout()
    plt.savefig(outpath, dpi=300)
    plt.show()
    print("Saved:", outpath)

In [ ]:
print(type(spleen_curves))
print(type(liver_curves))
print(spleen_curves.keys())
print(liver_curves.keys())

In [ ]:
# Adjust these paths to your actual folders
SPLEEN_IMG = "/content/data/Task09_Spleen/imagesTr"
SPLEEN_LAB = "/content/data/Task09_Spleen/labelsTr"

LIVER_IMG = "/content/data/Task03_Liver/imagesTr"
LIVER_LAB = "/content/data/Task03_Liver/labelsTr"

import os

def build_file_list(img_dir, lab_dir):
    img_files = sorted([
        os.path.join(img_dir, f)
        for f in os.listdir(img_dir)
        if f.endswith(".nii.gz") and not f.startswith("._")
    ])
    lab_files = sorted([
        os.path.join(lab_dir, f)
        for f in os.listdir(lab_dir)
        if f.endswith(".nii.gz") and not f.startswith("._")
    ])
    return img_files, lab_files

spleen_img_files, spleen_lab_files = build_file_list(SPLEEN_IMG, SPLEEN_LAB)
liver_img_files, liver_lab_files   = build_file_list(LIVER_IMG, LIVER_LAB)

print("Spleen cases:", len(spleen_img_files))
print("Liver cases:", len(liver_img_files))

In [ ]:
!ls /content/data

In [ ]:
!ls /content

In [19]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found
